In [ ]:
# EncoderGuiSmoke.ipynb
# Phase 7G+ rotary-encoder HDMI GUI control with live-apply.
# Single-cell notebook. Operates the AudioLabOverlay through the
# compact-v2 GUI; raw register dumps and ipywidgets sub-UIs have been
# removed on purpose.
#
# Operation (D47 / D51):
#   Encoder0 rotate              -> select effect (GUI EFFECTS order)
#   Encoder0 button-down edge OR
#   Encoder0 HW short_press      -> toggle ON/OFF of the selected effect
#                                   (D51 fallback catches taps shorter than
#                                    the poll period)
#   Encoder1 rotate (no hold)    -> select knob
#   Encoder1 rotate (hold)       -> cycle model index (OD / DIST / AMP / CAB)
#   Encoder2 rotate              -> change focused knob value (throttled apply)
#
# RAT (Distortion pedal-mask bit 2) is intentionally EXCLUDED from
# encoder-driven control. The Clash stage and notebook mirror API are
# untouched; only the encoder runtime skips RAT.
#
# Per-encoder SW polarity (D51 bench finding):
#   All three encoder modules on this rig are standard pull-up + push-to-
#   ground type: idle = HIGH, pressed = LOW. The D47 commit incorrectly
#   recorded these as "SW=HIGH when pressed" so the runner / earlier
#   notebooks defaulted to sw_active_low=False, which made every
#   encoder appear permanently pressed at idle. Setting
#   sw_active_low=True restores correct polarity for all three; no
#   software inversion is needed.
#
# Threading + render-cache model (D51 follow-up, latency fix):
#   The PIL-based compact-v2 renderer takes ~200-300 ms per cold frame
#   on the PYNQ-Z2 ARM Cortex-A9. Two changes keep it usable:
#     (a) render + write_frame run in a daemon thread so the encoder
#         poll loop is not blocked by PIL work.
#     (b) a single persistent `RenderCache` instance is passed to
#         every render call. Without this, the renderer builds a fresh
#         cache each call and every gradient / text glyph is rasterised
#         from scratch every frame. With the persistent cache warmed
#         up on the first render, subsequent frames hit `text_cache`
#         and `gradient_cache` and render several times faster.
#   The CPython GIL still serialises pure-Python work, so peak render
#   FPS is ultimately bounded by the cache-warm render time. The audio
#   path is unaffected -- apply runs from the main thread on each
#   encoder event with ~33 ms poll latency.
#
# Smoke baseline:
#   axi_encoder_input ip_dict key : enc_in_0/s_axi
#   EXPECTED_VERSION              : 0x00070001
#   CONFIG_DEFAULT                : 0x00010105 (low 16 bits asserted only)
#   VTC GEN_ACTSZ                 : 0x02580320  (SVGA 800x600 @ 40 MHz)
#
# HDMI PLL caveat (D25, memory: rgb2dvi-pll-edge-at-40mhz):
#   Phase 6I C2 runs HDMI at 40 MHz which sits at the rgb2dvi v1.4
#   kClkRange=3 VCO lower bound (800 MHz). A second `download=True`
#   in the same session (or after a kernel restart while the FPGA is
#   still programmed) can knock the PLL out and drop the LCD to
#   white. Below we use the HdmiGuiShow.ipynb smart-attach pattern.

import os
import sys
import time
import threading

PROJECT_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
for _p in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, "GUI")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pynq import PL
from audio_lab_pynq import AudioLabOverlay
from audio_lab_pynq.encoder_input import (
    EncoderInput,
    EncoderEvent,
    EXPECTED_VERSION,
    CONFIG_DEFAULT,
)
from audio_lab_pynq.encoder_ui import EncoderUiController
from audio_lab_pynq.encoder_effect_apply import EncoderEffectApplier
from audio_lab_pynq.hdmi_backend import (
    AudioLabHdmiBackend,
    DEFAULT_WIDTH,
    DEFAULT_HEIGHT,
)
from audio_lab_pynq.hdmi_state.resource_sampler import ResourceSampler
from GUI.compact_v2.state import AppState
from GUI.compact_v2.renderer import (
    render_frame_800x480_compact_v2,
    make_pynq_static_render_cache,
)


def _find_encoder_ip_name(overlay):
    ip_dict = getattr(overlay, "ip_dict", {})
    for name in ("enc_in_0/s_axi", "enc_in_0",
                 "axi_encoder_input_0", "axi_encoder_input"):
        if name in ip_dict:
            return name
    for name in ip_dict:
        lower = name.lower()
        if "encoder" in lower or lower.startswith("enc_in"):
            return name
    raise RuntimeError("encoder IP not found in overlay.ip_dict")


# --- Tunables -------------------------------------------------------------
GUI_SECONDS        = 600
POLL_HZ_ACTIVE     = 30
POLL_HZ_IDLE       = 10
IDLE_THRESHOLD_S   = 1.0
MAX_RENDER_FPS     = 15
STATUS_INTERVAL_S  = 2.0
LIVE_APPLY         = True
APPLY_INTERVAL_MS  = 50
VALUE_STEP         = 5.0
SKIP_RAT           = False
NO_AUDIO_APPLY     = False
# All three encoders on this rig: idle = HIGH, pressed = LOW. The IP
# CONFIG sw_active_low bit (active low = True) flips its internal
# polarity so BUTTON_STATE bit i = 1 means "encoder i pressed" and the
# HW short_press latch fires on actual press (HIGH -> LOW), not on
# release. Set this False only if your modules report HIGH when
# pressed.
SW_ACTIVE_LOW      = True
# Per-encoder button inversion mask. Bit i set = invert encoder i's
# button level in Python. Use only when one specific encoder differs
# in polarity from the global sw_active_low setting. With all three
# encoders matched on this rig the mask is 0.
BUTTON_INVERT_MASK = 0b000
RUN_ENCODER_GUI    = True


def stop_encoder_gui():
    global RUN_ENCODER_GUI
    RUN_ENCODER_GUI = False


def _fmt_bin3(value):
    return "{0:03b}".format(int(value) & 0x7)


class _EncoderButtonInvert(object):
    """Wrap an EncoderInput and selectively invert per-encoder button bits."""

    def __init__(self, inner, invert_mask):
        self._inner = inner
        self._mask = int(invert_mask) & 0x7

    def poll(self, *args, **kwargs):
        return self._inner.poll(*args, **kwargs)

    def read_button_state(self):
        return int(self._inner.read_button_state()) ^ self._mask

    def read_version(self):
        return self._inner.read_version()

    def read_config(self):
        return self._inner.read_config()

    def configure(self, **kwargs):
        return self._inner.configure(**kwargs)


# --- Overlay attach (smart, PLL-safe) ------------------------------------
if "ovl" in globals():
    print("ovl already attached in this kernel; reusing.")
else:
    try:
        loaded_path = PL.bitfile_name or ""
    except Exception:
        loaded_path = ""
    loaded_basename = os.path.basename(loaded_path)
    if loaded_basename == "audio_lab.bit":
        print("pynq.PL reports audio_lab.bit already loaded (%s)." % loaded_path)
        print("Attaching with download=False to preserve rgb2dvi PLL lock.")
        ovl = AudioLabOverlay(download=False)
    else:
        print("pynq.PL bitfile is %r; loading audio_lab.bit fresh." % loaded_path)
        ovl = AudioLabOverlay()

from pynq import MMIO as _MMIO
_vtc_desc = ovl.ip_dict["v_tc_hdmi"]
_vtc_actsz = int(_MMIO(int(_vtc_desc["phys_addr"]),
                       int(_vtc_desc["addr_range"])).read(0x60))
assert _vtc_actsz == 0x02580320, ("VTC GEN_ACTSZ = 0x%08X (expected "
                                  "0x02580320 SVGA 800x600 @ 40 MHz)"
                                  % _vtc_actsz)

# --- Encoder + GUI wiring -------------------------------------------------
_raw_enc = EncoderInput.from_overlay(ovl, ip_name=_find_encoder_ip_name(ovl))
_version = _raw_enc.read_version()
assert _version == EXPECTED_VERSION, (
    "encoder VERSION = 0x%08X expected 0x%08X" % (_version, EXPECTED_VERSION))
_raw_enc.configure(clear_on_read=True, sw_active_low=SW_ACTIVE_LOW)
_cfg = _raw_enc.read_config()
assert (_cfg & 0xFFFF) == (CONFIG_DEFAULT & 0xFFFF), (
    "encoder CONFIG = 0x%08X expected ~0x%08X" % (_cfg, CONFIG_DEFAULT))
print("encoder CONFIG = 0x%08X (sw_active_low=%s, button_invert_mask=0b%s)"
      % (_cfg, SW_ACTIVE_LOW, _fmt_bin3(BUTTON_INVERT_MASK)))

enc = _EncoderButtonInvert(_raw_enc, BUTTON_INVERT_MASK) \
    if BUTTON_INVERT_MASK else _raw_enc

state = AppState()
state.live_apply = bool(LIVE_APPLY)
state.apply_interval_ms = int(APPLY_INTERVAL_MS)

applier_overlay = None if NO_AUDIO_APPLY else ovl
applier = EncoderEffectApplier(
    applier_overlay,
    apply_interval_s=max(0.001, float(APPLY_INTERVAL_MS) / 1000.0),
    dry_run=bool(NO_AUDIO_APPLY),
    skip_rat=bool(SKIP_RAT),
)
controller = EncoderUiController(
    state, applier=applier, live_apply=bool(LIVE_APPLY),
    skip_rat=bool(SKIP_RAT), value_step=float(VALUE_STEP),
    apply_on_value_change=False,
)
backend = AudioLabHdmiBackend(ovl, width=DEFAULT_WIDTH, height=DEFAULT_HEIGHT)
backend.start()

# Single persistent RenderCache so text and gradient bitmaps survive
# between frames. The first render warms the cache (slow); every
# subsequent render hits the cache and is much faster.
_render_cache = make_pynq_static_render_cache(max_frame_entries=4)

_warm_t0 = time.time()
_initial_frame = render_frame_800x480_compact_v2(state, cache=_render_cache)
backend.write_frame(_initial_frame, placement="manual",
                    offset_x=0, offset_y=0)
_warm_dt = time.time() - _warm_t0
print("Initial render (cold cache): %.0f ms" % (_warm_dt * 1000.0))

_warm2_t0 = time.time()
_initial_frame2 = render_frame_800x480_compact_v2(state, cache=_render_cache)
_warm2_dt = time.time() - _warm2_t0
print("Second render (warm cache): %.0f ms" % (_warm2_dt * 1000.0))

print("Encoder GUI ready. live_apply=%s skip_rat=%s apply_interval_ms=%d"
      % (LIVE_APPLY, SKIP_RAT, APPLY_INTERVAL_MS))
print("Encoder0 rotate=effect select, button-down edge / short_press=toggle on/off")
print("Encoder1 rotate=knob select; PRESS+ROTATE=model cycle (OD/DIST/AMP/CAB)")
print("Encoder2 rotate=value change (live apply throttled)")
print("RAT pedal model is excluded from encoder control (skip_rat=%s)"
      % SKIP_RAT)


def _render_signature(s):
    knobs = ()
    akv = getattr(s, "all_knob_values", None)
    if isinstance(akv, dict):
        knobs = tuple((k, tuple(v)) for k, v in akv.items())
    return (
        getattr(s, "selected_effect", None),
        getattr(s, "selected_knob", None),
        bool(getattr(s, "value_dirty", False)),
        bool(getattr(s, "apply_pending", False)),
        bool(getattr(s, "edit_mode", False)),
        bool(getattr(s, "model_select_mode", False)),
        getattr(s, "dist_model_idx", None),
        getattr(s, "amp_model_idx", None),
        getattr(s, "cab_model_idx", None),
        bool(getattr(s, "last_apply_ok", True)),
        str(getattr(s, "last_apply_message", "")),
        id(getattr(s, "last_encoder_event", None)),
        tuple(bool(v) for v in (getattr(s, "effect_on", []) or [])),
        knobs,
    )


def _fmt_pct(v):
    return ("%5.1f%%" % v) if v is not None else "  n/a"


def _fmt_temp(v):
    return ("%4.1fC" % v) if v is not None else "  n/a"


# --- Render thread -------------------------------------------------------
_render_request_sig = [None]
_render_done_sig    = [None]
_render_alive       = [True]
_render_frames      = [0]
_render_errors      = [0]
_render_last_t      = [0.0]
_render_last_dt_ms  = [0.0]
_render_min_period  = 1.0 / float(MAX_RENDER_FPS)


def _render_worker():
    while _render_alive[0]:
        target = _render_request_sig[0]
        if target is None or target == _render_done_sig[0]:
            time.sleep(0.005)
            continue
        now = time.time()
        wait = _render_min_period - (now - _render_last_t[0])
        if wait > 0:
            time.sleep(wait)
        t0_r = time.time()
        try:
            frame = render_frame_800x480_compact_v2(state, cache=_render_cache)
            backend.write_frame(frame, placement="manual",
                                offset_x=0, offset_y=0)
            _render_frames[0] += 1
        except Exception as exc:
            _render_errors[0] += 1
            print("[render] err:", repr(exc))
        _render_done_sig[0] = target
        _render_last_t[0] = time.time()
        _render_last_dt_ms[0] = (_render_last_t[0] - t0_r) * 1000.0


_render_thread = threading.Thread(target=_render_worker, daemon=True)
_render_thread.start()
print("Render thread started (MAX_RENDER_FPS=%d, daemon)" % MAX_RENDER_FPS)


sampler = ResourceSampler()
sampler.sample()

poll_period_active = 1.0 / float(POLL_HZ_ACTIVE)
poll_period_idle   = 1.0 / float(POLL_HZ_IDLE)

t0 = time.time()
last_event_t       = t0
last_sig           = None
last_status_t      = 0.0
last_status_frames = 0
last_status_polls  = 0
polls              = 0

try:
    while RUN_ENCODER_GUI and (time.time() - t0) < GUI_SECONDS:
        loop_t = time.time()
        polls += 1
        n_events = controller.tick(enc, timestamp=loop_t - t0)
        if n_events:
            last_event_t = loop_t

        sig = _render_signature(state)
        if sig != last_sig:
            state.t = loop_t - t0
            _render_request_sig[0] = sig
            last_sig = sig

        if (loop_t - last_status_t) >= STATUS_INTERVAL_S:
            r = sampler.sample()
            dt = ((loop_t - last_status_t) if last_status_t > 0
                  else max(1e-3, loop_t - t0))
            frames_now = _render_frames[0]
            render_fps = (frames_now - last_status_frames) / dt
            poll_hz = (polls - last_status_polls) / dt
            rss_mb = int(r.get("proc_rss_kb", 0)) // 1024
            mem_total_mb = int(r.get("mem_total_kb", 0)) // 1024
            mem_avail_mb = int(r.get("mem_avail_kb", 0)) // 1024
            mem_used_mb = (mem_total_mb - mem_avail_mb
                           if mem_total_mb else 0)
            mem_used_pct = (100.0 * mem_used_mb / mem_total_mb)                 if mem_total_mb else None
            idle_for = loop_t - last_event_t
            mode = "idle" if idle_for > IDLE_THRESHOLD_S else "active"
            msg = applier.last_apply_message or "-"
            if len(msg) > 28:
                msg = msg[:25] + "..."
            print(
                "t=%6.1fs mode=%-6s poll=%5.1fHz render=%4.1ffps "
                "rdt=%4.0fms sys_cpu=%s proc_cpu=%s mem=%4d/%4dMB(%s) "
                "rss=%4dMB temp=%s fx=%d knob=%d live=%s apply=%s "
                "last=%s sw=%d%d%d rerr=%d"
                % (
                    loop_t - t0, mode, poll_hz, render_fps,
                    _render_last_dt_ms[0],
                    _fmt_pct(r.get("sys_cpu_pct")),
                    _fmt_pct(r.get("proc_cpu_pct")),
                    mem_used_mb, mem_total_mb, _fmt_pct(mem_used_pct),
                    rss_mb, _fmt_temp(r.get("temperature_c")),
                    state.selected_effect, state.selected_knob,
                    "ON" if state.live_apply else "off",
                    "OK" if applier.last_apply_ok else "ERR",
                    msg,
                    int(controller._current_pressed[0]),
                    int(controller._current_pressed[1]),
                    int(controller._current_pressed[2]),
                    _render_errors[0],
                )
            )
            last_status_t = loop_t
            last_status_frames = frames_now
            last_status_polls = polls

        idle_for = loop_t - last_event_t
        period = (poll_period_idle if idle_for > IDLE_THRESHOLD_S
                  else poll_period_active)
        elapsed = time.time() - loop_t
        if elapsed < period:
            time.sleep(period - elapsed)
except KeyboardInterrupt:
    pass
finally:
    _render_alive[0] = False
    _render_thread.join(timeout=2.0)
    backend.stop()
